# NB05 · Recuperación: una sola puerta de entrada

NB04 dejó un índice de 15.000 puntos en Qdrant. Este notebook le pone la puerta: **una función que recibe una consulta en texto y devuelve resultados normalizados**, que es lo que §3.3 pide con esas palabras.

Lo que se demuestra no es que el buscador encuentre cosas buenas —eso es calidad, y se mide en NB09—, sino que **el sistema se comporta como dice**: que el filtro lo ejecuta la base y no Python, que una respuesta vacía y un fallo salen por canales distintos, y que los cuatro casos borde del enunciado están tratados a propósito.

> 📋 **Se demuestra enseñando lo que sale**: cada sección trae los `product_id`, los títulos y los scores que devolvió el motor, no un veredicto sobre ellos. Un ✅ es una afirmación del código sobre sí mismo; las filas son la prueba con la que cualquiera puede contradecirlo.
>
> Y sobre **varias consultas**: seis de demostración —tres formulaciones de la misma necesidad y tres de cliente— más las cuatro filtradas del §5. Con una sola frase, ninguna tabla distingue un caso borde bien tratado de una consulta que no casaba con nada.

### Lo que se hereda y no se vuelve a decidir

| | |
|---|---|
| Motor | Qdrant (R03) |
| Colección | `aurum_catalogo__gemini_embedding_2__A4__768` (NB04 § G) |
| Modelo · plantilla · dim | `gemini-embedding-2` [sin_contrato] · A4 · 768 |
| Normalización del filtro | `casefold_unaccent` al buscar, el dato crudo guardado (D03) |

**Nada de ANN se toca aquí.** Los parámetros del índice son los de por defecto y su estudio es NB06, con D16 fijada antes de ver la curva.

### Las cuatro decisiones de este notebook

| | Decidido | Por qué |
|---|---|---|
| Forma del resultado | `Resultado` **hereda** de `SearchResult` | Un `Resultado` *es* un `SearchResult`, así que NB09 lo mezcla con el baseline léxico sin traducir |
| Colección vacía | Lista vacía | *"A nivel de usuario, cuando busque, no le aparecerá nada si no se encuentra nada"* |
| Timeout | **30 s** y excepción controlada | *"No es el mismo caso que no haber encontrado resultados"* |
| Color en la interfaz | **No** | Queda como capacidad del almacén; el enunciado solo pide marca |

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000) · las tres familias de consultas
import os
import sys
from functools import lru_cache, partial
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

from dotenv import load_dotenv

from aurum.almacen import filter_reach
from aurum.busqueda import (
    BuscadorVectorial,
    auditar_casos_borde,
    auditar_filtro_de_marca,
    auditar_forma_de_los_resultados,
    auditar_post_filtro,
    auditar_variantes_de_marca,
    solapamiento_entre_consultas,
    tabla_de_resultados,
)
from aurum.datos import load_csv
from aurum.embeddings import GeminiEncoder, encode_corpus, truncate_dim
from aurum.motores import CATALOG_PREFIX, Point, catalog_collection_name
from aurum.motores.qdrant import QdrantStore

load_dotenv(Path("..") / ".env")
DATA, CACHE = Path("..") / "data", Path("..") / "artifacts" / "embeddings"
completo = load_csv(DATA / "catalogo_productos.csv")
filtradas = load_csv(DATA / "consultas_filtradas.csv")
evaluacion = load_csv(DATA / "consultas_evaluacion.csv")
desarrollo = load_csv(DATA / "consultas_desarrollo.csv")

MODELO, CONTRATO, PLANTILLA = "gemini-embedding-2", "sin_contrato", "A4"
DIM, TOP_K = 768, 10
TIMEOUT_S = 30            # decision de NB05
COLECCION = catalog_collection_name(model=MODELO, template=PLANTILLA, dim=DIM)

# Las seis consultas con las que se ejercita la interfaz. No son inventadas:
# salen de los CSV del enunciado. Las tres primeras son la MISMA necesidad
# escrita de tres formas (§5 pide consultas de distinto tipo) y las tres
# últimas son consultas de cliente, con sus faltas y sus negaciones.
ORDEN_TIPO = {"direct": 0, "semantic": 1, "context": 2}
TRES_FORMULACIONES = [
    (f["evaluation_id"], f["query_text"])
    for f in sorted(
        (f for f in evaluacion.to_dict("records")
         if f["evaluation_id"].startswith("EVAL-100455")),
        key=lambda f: ORDEN_TIPO[f["query_type"]],
    )
]
DE_CLIENTE = [
    (f["workload_id"], f["query_text"])
    for f in desarrollo.to_dict("records")
    if f["workload_id"] in ("DEV-38249", "DEV-43240", "DEV-61533")
]
DEMO = TRES_FORMULACIONES + DE_CLIENTE
POR_CASO = dict(DEMO)

print(f"coleccion: {COLECCION}")
print(f"catalogo : {len(completo)} productos · {len(filtradas)} consultas filtradas")
for caso, texto in DEMO:
    print(f"  {caso:22s} {texto}")

c:\Users\asus\Master\modulos\modulo10_bbdd\practica\AURUM_MARKET\aurum-market-catalog\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


coleccion: aurum_catalogo__gemini_embedding_2__A4__768
catalogo : 15000 productos · 4 consultas filtradas
  EVAL-100455-direct     taladro 24v batería
  EVAL-100455-semantic   quiero una herramienta inalámbrica potente para perforar sin depender de un enchufe
  EVAL-100455-context    taladro sin cable de 24 voltios que venga con su batería
  DEV-38249              estantes sin taladro habitacion
  DEV-43240              funda ipad air 4 sin tapa
  DEV-61533              lentejas sin gluten


## A · El buscador

Tres piezas y una decisión de diseño en cada una.

**El codificador se inyecta.** `aurum.busqueda` no sabe de Gemini: recibe una función que convierte texto en vector. Eso es lo que permite probar el buscador entero sin red —los 41 tests del módulo no tocan la API ni Docker— y lo que hará que cambiar de modelo no toque este fichero.

**Y se memoiza.** El contrato de §3.3 es *recibe una consulta en texto*, así que `buscar()` codifica; pero cada consulta se lanza muchas veces —la sección C repite las filtradas seis veces con distintos tamaños de recuperación— y pagar la misma llamada de API una y otra vez sería tirar dinero por un detalle de implementación. `lru_cache` lo evita sin cambiar el contrato, y la caché en disco hace que la segunda ejecución no pague ninguna.

**El timeout lo aplica el cliente, no el buscador.** Quien tiene el socket es el SDK, así que los 30 s se declaran al construir `QdrantStore`. El buscador los conoce solo para poder decir en el mensaje de error contra qué límite se agotó.

In [ ]:
# 📄 DATOS · las 4 consultas ya codificadas en NB04 (caché de artifacts/embeddings)
_encoder = GeminiEncoder(
    api_key=os.environ.get("GEMINI_API_KEY"), model_id=MODELO,
    native_dim=3072, window=8192,
)


@lru_cache(maxsize=256)
def codificar_consulta(texto: str):
    """Texto -> vector de 768, pagando la API una sola vez por consulta."""
    # `corpus_id` constante a propósito: la clave de caché incluye el
    # SHA del texto, así que ya distingue una consulta de otra. Meter el
    # texto en el nombre del fichero lo rompería con la primera consulta
    # que llevara una barra o un acento.
    codificado = encode_corpus(
        _encoder, [texto], corpus_id="consulta_suelta",
        kind="query", contract=CONTRATO, batch_size=1, cache_dir=CACHE,
    )
    return truncate_dim(codificado.vectors, DIM)[0]


almacen = QdrantStore(
    collection=COLECCION,
    url=os.environ.get("AURUM_QDRANT_URL", "http://localhost:6333"),
    api_key=os.environ.get("AURUM_QDRANT_API_KEY"),
    prefix=CATALOG_PREFIX,
    timeout=TIMEOUT_S,        # el limite lo aplica el cliente
)
buscador = BuscadorVectorial(
    almacen, codificar_consulta, top_k=TOP_K, timeout_s=TIMEOUT_S,
)

print(f"puntos en la coleccion: {almacen.count():,}".replace(",", "."))
print(f"indice al dia         : {almacen.index_ready()}")

puntos en la coleccion: 15.000
indice al dia         : True


### Seis consultas, y lo que devuelve cada una

Antes de medir nada, **verlo**. Las seis salen de los CSV del enunciado y están elegidas para que no se parezcan entre sí:

| Consultas | De dónde | Qué ponen a prueba |
|---|---|---|
| `EVAL-100455` ×3 | `consultas_evaluacion.csv` | La **misma necesidad** escrita como palabras clave (`direct`), como frase natural (`semantic`) y como situación (`context`) |
| `DEV-38249` · `DEV-43240` · `DEV-61533` | `consultas_desarrollo.csv` | Consultas **de cliente**: sin acentos, con negación (*sin taladro*, *sin tapa*) y una de categoría lejana (*lentejas sin gluten*) |

Una fila por producto recuperado, con los **5 primeros de cada consulta** para que quepa; el top-10 completo se recupera igual y es el que usan las secciones siguientes.

Qué mirar aquí, que no es la calidad —eso es NB09—:

- Que **las seis devuelven diez**, incluida `lentejas sin gluten`: un buscador denso siempre devuelve `k` vecinos, no existe el "no hay nada parecido". Que lo devuelto sea relevante es otra pregunta.
- Que la columna del score se llama `score_mayor_mejor`. El nombre lo pone la propia tabla a partir de lo que declara el motor: si esto fuera Weaviate saldría `score_menor_mejor` y el orden se leería al revés.

> 💸 La primera ejecución paga seis llamadas de codificación (una por consulta) y las guarda en `artifacts/embeddings`. Las siguientes no pagan ninguna.

In [ ]:
# 📄 DATOS · las 6 consultas de DEMO, del catálogo indexado en NB04
demo = [(caso, texto, buscador.buscar(texto, top_k=TOP_K)) for caso, texto in DEMO]

tabla_de_resultados(demo, top=5).style.hide(axis="index").set_properties(
    **{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"}
)

caso,consulta,posicion,product_id,marca,titulo,score_mayor_mejor
EVAL-100455-direct,taladro 24v batería,1,B09BQTS4FF,MAXDONE,"Taladro Atornillador 21V Taladro Percutor a Batería, 36 Nm…",0.610500
EVAL-100455-direct,taladro 24v batería,2,B07JHCZ1T4,FLY BIZ,"Flybiz 1650/min Taladro Atornillador 21V, Broca，Taladro sin…",0.603700
EVAL-100455-direct,taladro 24v batería,3,B09CYWTYVD,Mimajor,"Taladro Atornillador 12.6V, Destornillador Electrico con 2…",0.590300
EVAL-100455-direct,taladro 24v batería,4,B07ZCKTM2R,Naroote,Llave de impacto sin escobillas alimentada por batería de l…,0.589200
EVAL-100455-direct,taladro 24v batería,5,B07QY6R25X,Joiry,Joiry 24V 3.5Ah Batería para Bosch GBH24VRE GBH24VFR GBH24V…,0.588800
EVAL-100455-semantic,quiero una herramienta inalámbrica potente para perforar sin depender de un enchufe,1,B01A5VQHBY,WORX,Taladro Percutor Brushless 20V Worx WX373,0.642100
EVAL-100455-semantic,quiero una herramienta inalámbrica potente para perforar sin depender de un enchufe,2,B07JHCZ1T4,FLY BIZ,"Flybiz 1650/min Taladro Atornillador 21V, Broca，Taladro sin…",0.626500
EVAL-100455-semantic,quiero una herramienta inalámbrica potente para perforar sin depender de un enchufe,3,B09BQTS4FF,MAXDONE,"Taladro Atornillador 21V Taladro Percutor a Batería, 36 Nm…",0.625100
EVAL-100455-semantic,quiero una herramienta inalámbrica potente para perforar sin depender de un enchufe,4,B00G7614BK,DeWalt,DeWalt DCD795D2-QW - Taladro Percutor a bateria sin escobil…,0.621200
EVAL-100455-semantic,quiero una herramienta inalámbrica potente para perforar sin depender de un enchufe,5,B0071T3MOO,Bosch Professional,"Bosch 12V System GSB 12V-15 - Taladro Percutor a Batería, 3…",0.610000


### Un resultado, por dentro

La tabla de arriba aplana; el objeto tiene más. Los campos son los que exige §3.3 —`product_id`, posición, título, metadatos y score— y dos detalles que conviene mirar:

- **Los dos identificadores.** `document_id` es el `product_id`, que es lo que juzgan los qrels y lo que piden los CSV de entrega; `record_id` es el id del punto en Qdrant. No son intercambiables.
- **`score_es_similitud`** viene del motor, no de una suposición. Qdrant devuelve similitud; Weaviate habría devuelto distancia y el orden se leería al revés.

In [ ]:
caso_ejemplo, texto_ejemplo, resultados_ejemplo = demo[1]   # la formulación semántica
print(f"{caso_ejemplo}: {texto_ejemplo!r}\n")
for r in resultados_ejemplo[:3]:
    print(f"{r.rank}. {r.document_id}  score={r.score:.4f}  "
          f"similitud={r.score_es_similitud}")
    print(f"   {r.titulo[:80]}")
    print(f"   record_id={r.record_id} · marca={r.metadatos.get('brand') or '(vacía)'}")
    print(f"   payload: {sorted(r.metadatos)}")

EVAL-100455-semantic: 'quiero una herramienta inalámbrica potente para perforar sin depender de un enchufe'

1. B01A5VQHBY  score=0.6421  similitud=True
   Taladro Percutor Brushless 20V Worx WX373
   record_id=2a5aa063-7d33-5243-9ec2-3687ace89a66 · marca=WORX
   payload: ['active', 'brand', 'brand_normalized', 'catalog_version', 'color', 'color_normalized', 'product_id', 'title']
2. B07JHCZ1T4  score=0.6265  similitud=True
   Flybiz 1650/min Taladro Atornillador 21V, Broca，Taladro sin cable con luces led,
   record_id=ed4e0048-2c39-5074-87bd-38512d6eb0d8 · marca=FLY BIZ
   payload: ['active', 'brand', 'brand_normalized', 'catalog_version', 'color', 'color_normalized', 'product_id', 'title']
3. B09BQTS4FF  score=0.6251  similitud=True
   Taladro Atornillador 21V Taladro Percutor a Batería, 36 Nm Máx 2x1500mah Batería
   record_id=d0b2bccc-30f2-582c-b638-8a977b9537d9 · marca=MAXDONE
   payload: ['active', 'brand', 'brand_normalized', 'catalog_version', 'color', 'color_normalized', 'prod

### La misma necesidad, escrita de tres formas

Las tres primeras piden lo mismo —un taladro inalámbrico de 24 V con batería— con palabras muy distintas: si la interfaz fuera léxica, `taladro 24v batería` y *"quiero una herramienta inalámbrica potente para perforar sin depender de un enchufe"* no compartirían casi nada, porque **no comparten casi ninguna palabra**.

La tabla cuenta cuántos `product_id` comparten sus top-10 y si coinciden en el primero: cuanto más alto el solapamiento, **menos depende el resultado de cómo se escriba la consulta**. No es calidad —tres formulaciones podrían coincidir en diez resultados igual de malos—, es que la puerta responda igual ante las tres.

In [ ]:
solapamiento_entre_consultas(demo[:3]).style.hide(axis="index").set_properties(
    **{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"}
)

consulta_a,consulta_b,texto_a,texto_b,en_comun,de,solapamiento,mismo_primero
EVAL-100455-direct,EVAL-100455-semantic,taladro 24v batería,quiero una herramienta inalámbrica potente p…,4,10,40 %,False
EVAL-100455-direct,EVAL-100455-context,taladro 24v batería,taladro sin cable de 24 voltios que venga co…,9,10,90 %,False
EVAL-100455-semantic,EVAL-100455-context,quiero una herramienta inalámbrica potente p…,taladro sin cable de 24 voltios que venga co…,4,10,40 %,False


---

## B · Las cuatro consultas filtradas (§5)

El §8 lo pide como criterio de corrección: *"las consultas filtradas nunca devuelven otra marca"*. Medir solo la pureza no basta, por el motivo que ya apareció en NB04: **una respuesta vacía la cumple de forma vacía**, y un filtro roto que no devuelve nada saca el mismo 100 % que uno perfecto.

Por eso la tabla lleva el oráculo al lado —cuántos productos de esa marca hay en el catálogo, contados con pandas y sin motor—:

| Columna | Qué dice |
|---|---|
| `n_en_catalogo` | El oráculo. Distingue un cero legítimo de un filtro roto |
| `de_la_marca` | Cuántos de los devueltos son de verdad de esa marca, auditado contra `brand_normalized` |
| `pureza` | El criterio del §8 |
| `veredicto` | Cruza las dos cosas |

Hay un tercer veredicto que interesa especialmente: **cobertura corta**. Las cuatro marcas tienen más de 10 productos, así que las cuatro deben devolver 10; si alguna devuelve menos, lo más probable es que se esté filtrando después en Python en vez de en la base — justo lo que mide la sección C.

Debajo del resumen van **los 40 resultados**, uno por fila: la tabla de arriba dice *"pureza 100 %"* y la de abajo permite comprobarlo sin fiarse.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv — el oráculo, contado sin motor
# `unaccent` y no `raw`: el motor filtra contra `brand_normalized`, así que
# el oráculo tiene que contar sobre la misma clave o compararíamos cosas
# distintas.
alcance = filter_reach(
    completo, filtradas["filter_value"].tolist(), field="brand",
    modes=("unaccent",),
)
ORACULO = dict(zip(alcance["filtro"], alcance["n_productos"]))
print("productos por marca en el catálogo:", ORACULO)

pureza = auditar_filtro_de_marca(
    buscador, filtradas.to_dict("records"), alcance=ORACULO, top_k=TOP_K
)
pureza.style.hide(axis="index").set_properties(
    **{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"}
)

productos por marca en el catálogo: {'Einhell': 30, 'Apple': 100, 'NIKE': 295, 'SAMSUNG': 155}


caso,consulta,marca,n_en_catalogo,n_resultados,de_la_marca,pureza,veredicto
FILTER-001,herramienta inalámbrica para perforar,Einhell,30,10,10,100%,✅ pureza 100 % y cobertura completa (10 de 10)
FILTER-002,tableta ligera para estudiar y tomar apuntes,Apple,100,10,10,100%,✅ pureza 100 % y cobertura completa (10 de 10)
FILTER-003,zapatillas cómodas para salir a correr,NIKE,295,10,10,100%,✅ pureza 100 % y cobertura completa (10 de 10)
FILTER-004,monitor para trabajar con varias ventanas,SAMSUNG,155,10,10,100%,✅ pureza 100 % y cobertura completa (10 de 10)


#### Los 40 resultados, para poder contradecir la tabla anterior

Una fila por producto devuelto en las cuatro consultas filtradas. `marca` es el valor **crudo** que se guardó en el payload y `marca_normalizada` es la clave contra la que filtra el motor (D03: se guarda tal cual y se normaliza al buscar).

Juntas hacen visible el filtro: `NIKE` y `Nike` son marcas distintas para un `equals` sobre el valor crudo y **la misma** para el que se ejecuta de verdad. Si en `marca_normalizada` apareciera un solo valor distinto del pedido, el 100 % de arriba sería mentira.

**Y las cuatro piden su marca escrita a propósito de otra forma**: el catálogo guarda `NIKE` y `SAMSUNG` en mayúsculas, pero se piden `einhell`, `  APPLE `, `Nike` y `Sámsung`. Si el filtro solo sirviera para quien escribe la marca igual que el CSV, estas cuarenta filas estarían vacías — y la pureza **seguiría dando 100 %**.

In [ ]:
# 📄 DATOS · 📚 consultas_filtradas.csv — las 4 consultas del §5, con filtro nativo
# Cada una pide su marca escrita como la escribiría un usuario, no como
# está en el catálogo: minúsculas, mayúsculas con espacios, capitalizada y
# con un acento colado. Las cuatro tienen que devolver lo mismo.
ESCRITURAS = {
    "Einhell": "einhell",
    "Apple": "  APPLE ",
    "NIKE": "Nike",
    "SAMSUNG": "Sámsung",
}
resultados_filtrados = [
    (
        f["workload_id"],
        f"{f['query_text']}  ·  marca pedida={ESCRITURAS[f['filter_value']]!r}",
        buscador.buscar(
            f["query_text"], top_k=TOP_K, marca=ESCRITURAS[f["filter_value"]]
        ),
    )
    for f in filtradas.to_dict("records")
]

detalle_filtradas = tabla_de_resultados(
    resultados_filtrados,
    metadatos={"brand": "marca", "brand_normalized": "marca_normalizada"},
)
detalle_filtradas.style.hide(axis="index").set_properties(
    **{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"}
)

caso,consulta,posicion,product_id,marca,marca_normalizada,titulo,score_mayor_mejor
FILTER-001,herramienta inalámbrica para perforar · marca pedida='einhell',1,B07GSD93Q8,Einhell,einhell,"Einhell Taladro de impacto sin cable TE-CD 12/1 Li-i (2x 2,…",0.534800
FILTER-001,herramienta inalámbrica para perforar · marca pedida='einhell',2,B08ZMZZRJH,Einhell,einhell,Einhell Kit Martillo perforador con batería HEROCCO+5 Power…,0.524200
FILTER-001,herramienta inalámbrica para perforar · marca pedida='einhell',3,B01N6Y6G16,Einhell,einhell,Einhell Atornillador inalámbrico TE-CD 18 Li-i Brushless Po…,0.516800
FILTER-001,herramienta inalámbrica para perforar · marca pedida='einhell',4,B01AB2KU5G,Einhell,einhell,Einhell Expert Martillo perforador y cincelador Power X-Cha…,0.514000
FILTER-001,herramienta inalámbrica para perforar · marca pedida='einhell',5,B00IYEEY0Q,Einhell,einhell,"Einhell Martillo perforador TC-RH 900 (900 W, 3 J, capacida…",0.470600
FILTER-001,herramienta inalámbrica para perforar · marca pedida='einhell',6,B01AB2KSYE,Einhell,einhell,Einhell Original Batería del sistema Power X-Change (baterí…,0.465400
FILTER-001,herramienta inalámbrica para perforar · marca pedida='einhell',7,B0798C8QF6,Einhell,einhell,"Einhell Herramienta multifuncional TC-MG 220/1 E (220 W, in…",0.464700
FILTER-001,herramienta inalámbrica para perforar · marca pedida='einhell',8,B00G66VIOY,Einhell,einhell,"Einhell Lijadora Delta TC-DS 19 (190W, 20000 rpm, empuñadur…",0.464000
FILTER-001,herramienta inalámbrica para perforar · marca pedida='einhell',9,B01MYUJ1A8,Einhell,einhell,"Einhell Cepillo eléctrico con cable - TC-PL 750. 750W, 240…",0.443500
FILTER-001,herramienta inalámbrica para perforar · marca pedida='einhell',10,B08LQGRNQY,Einhell,einhell,"Einhell Compresor TE-AC 50 Silent (compresor susurrante, 1.…",0.438900


#### La misma marca, escrita de todas las formas en que se escribe

La celda anterior usa una escritura distinta por consulta; esta las barre todas. Cada fila es **la misma consulta filtrada por la misma marca**, cambiando solo cómo se teclea, y se compara con lo que devolvió la escritura del CSV.

| Columna | Qué dice |
|---|---|
| `marca_pedida` | Lo que se teclea, entre comillas para que se vean los espacios |
| `viaja_al_motor` | Lo que sale de `normalize_brand` y llega a la base. **Aquí es donde ocurre D03** |
| `iguales_a_la_canonica` | Cuántos de los diez coinciden con los de la escritura del CSV, y si además en el mismo orden |
| `esperado` | `los mismos 10` para las seis primeras; `0` para las dos últimas |

Las dos últimas variantes de cada marca son **faltas de verdad** —un espacio de más dentro y una letra de menos— y tienen que devolver cero. Están para marcar el límite: normalizar iguala la caja y los acentos, **no corrige faltas**. Sin ellas, una tabla en la que todo devuelve diez no distinguiría un filtro que normaliza de uno que casa con cualquier cosa.

> 🎯 Lo que esta tabla caza es un fallo caro y silencioso. Si la normalización fuera `casefold` en vez de `unaccent`, `Sámsung` viajaría como `sámsung`, no casaría con nada, y **la pureza del §8 seguiría saliendo al 100 %** mientras el usuario ve una lista vacía. Ese es el caso que ningún resumen de veredictos enseña.

In [ ]:
# 📄 DATOS · 📚 consultas_filtradas.csv — 4 marcas × sus escrituras
variantes = auditar_variantes_de_marca(
    buscador, filtradas.to_dict("records"), top_k=TOP_K
)
variantes.style.hide(axis="index").set_properties(
    **{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"}
)

marca_en_el_csv,variante,marca_pedida,viaja_al_motor,n_resultados,de_la_marca,iguales_a_la_canonica,esperado,veredicto
Einhell,tal como está en el CSV,'Einhell','einhell',10,10,10 de 10 · mismo orden,los mismos 10,✅ idéntica a la canónica
Einhell,todo en minúsculas,'einhell','einhell',10,10,10 de 10 · mismo orden,los mismos 10,✅ idéntica a la canónica
Einhell,TODO EN MAYÚSCULAS,'EINHELL','einhell',10,10,10 de 10 · mismo orden,los mismos 10,✅ idéntica a la canónica
Einhell,con espacios alrededor,' Einhell ','einhell',10,10,10 de 10 · mismo orden,los mismos 10,✅ idéntica a la canónica
Einhell,con un acento colado,'Éinhell','einhell',10,10,10 de 10 · mismo orden,los mismos 10,✅ idéntica a la canónica
Einhell,con un espacio dentro,'Ei nhell','ei nhell',0,0,0 de 10,0: no es esa marca,"✅ no casa, y no debía"
Einhell,con una letra de menos,'Einhel','einhel',0,0,0 de 10,0: no es esa marca,"✅ no casa, y no debía"
Apple,tal como está en el CSV,'Apple','apple',10,10,10 de 10 · mismo orden,los mismos 10,✅ idéntica a la canónica
Apple,todo en minúsculas,'apple','apple',10,10,10 de 10 · mismo orden,los mismos 10,✅ idéntica a la canónica
Apple,TODO EN MAYÚSCULAS,'APPLE','apple',10,10,10 de 10 · mismo orden,los mismos 10,✅ idéntica a la canónica


---

## C · Filtro nativo contra post-filtro

La decisión de filtrar en la base y no en Python estaba tomada desde NB04 —es lo que el enunciado exige—, pero hasta ahora se sostenía en un argumento aritmético: **`Einhell` son 30 productos de 15.000, el 0,2 %**, así que para esperar diez suyos habría que recuperar del orden de 5.000 candidatos y descartar 4.990.

La celda lo mide en vez de suponerlo, y **en las cuatro marcas, no solo en `Einhell`**: con una sola la conclusión dependería de cuál se eligiera, porque la del enunciado es la más rara y no llega a diez ni con mil candidatos, mientras que una marca frecuente sí llega y haría parecer que el post-filtro *funciona con un poco de sobre-recuperación*.

Cada bloque de seis filas es una marca: la primera es el filtro nativo y las cinco siguientes recuperan sin filtrar `10 × factor` candidatos y se quedan con los de la marca.

| Columna | Cómo se lee |
|---|---|
| `n_en_catalogo` · `pct_del_catalogo` | Cuántos productos de esa marca hay. **Cuanto más bajo, antes falla el post-filtro** |
| `candidatos` | Cuántos productos trajo el motor en esa estrategia |
| `de_la_marca` | Cuántos de esos candidatos eran de la marca pedida |
| `descartados` | Los que Python tendría que tirar. Es el coste, con la resta hecha |
| `llega_a_10` | Si la estrategia consigue el top-10 que pidió el usuario |
| `ms` | Lo que tardó. El filtro nativo trae 10 y ya; el post-filtro paga por traer basura |

> Lo que hay que buscar no es que el post-filtro sea más lento —que lo es—, sino **dónde falla**: si `llega_a_10` sigue en `False` con ×100 para la marca rara mientras las frecuentes lo consiguen, no es una alternativa peor, es que **falla justo donde el filtro hace falta**.

In [ ]:
# 📄 DATOS · 📚 consultas_filtradas.csv — las 4 marcas × 6 estrategias
for caso in filtradas.to_dict("records"):
    n_marca = ORACULO[caso["filter_value"]]
    print(f"{caso['workload_id']}: marca {caso['filter_value']!r} · "
          f"{n_marca} productos de {len(completo)} "
          f"({100 * n_marca / len(completo):.2f} % del catálogo)")

comparativa = auditar_post_filtro(
    buscador, filtradas.to_dict("records"), alcance=ORACULO,
    top_k=TOP_K, factores=(1, 5, 10, 50, 100), n_catalogo=len(completo),
)
comparativa.style.hide(axis="index").set_properties(
    **{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"}
)

FILTER-001: marca 'Einhell' · 30 productos de 15000 (0.20 % del catálogo)
FILTER-002: marca 'Apple' · 100 productos de 15000 (0.67 % del catálogo)
FILTER-003: marca 'NIKE' · 295 productos de 15000 (1.97 % del catálogo)
FILTER-004: marca 'SAMSUNG' · 155 productos de 15000 (1.03 % del catálogo)


caso,marca,n_en_catalogo,pct_del_catalogo,estrategia,candidatos,de_la_marca,descartados,llega_a_10,ms
FILTER-001,Einhell,30,0.20 %,filtro nativo (top_k=10),10,10,0,True,6.800000
FILTER-001,Einhell,30,0.20 %,post-filtro ×1,10,2,8,False,38.200000
FILTER-001,Einhell,30,0.20 %,post-filtro ×5,50,4,46,False,27.700000
FILTER-001,Einhell,30,0.20 %,post-filtro ×10,100,4,96,False,36.700000
FILTER-001,Einhell,30,0.20 %,post-filtro ×50,500,8,492,False,304.700000
FILTER-001,Einhell,30,0.20 %,post-filtro ×100,1000,13,987,True,290.600000
FILTER-002,Apple,100,0.67 %,filtro nativo (top_k=10),10,10,0,True,9.700000
FILTER-002,Apple,100,0.67 %,post-filtro ×1,10,3,7,False,15.300000
FILTER-002,Apple,100,0.67 %,post-filtro ×5,50,22,28,True,12.400000
FILTER-002,Apple,100,0.67 %,post-filtro ×10,100,28,72,True,18.100000


---

## D · Los cuatro casos borde

El enunciado los exige por escrito: *"tratamiento explícito de colecciones vacías, filtros sin resultados y proveedores no disponibles"*, más el `top_k` mayor que el número de puntos que añade el plan.

| Caso | Qué debe pasar | Por qué se decidió así |
|---|---|---|
| Colección vacía | Lista vacía | Decisión de NB05: el usuario no ve nada, y no ver nada no es un error |
| Filtro sin resultados | Lista vacía | Lo impone el enunciado, no se decide |
| Motor no disponible | `MotorNoDisponible` | Es un fallo, no una respuesta: mezclarlo con la lista vacía haría creer que el catálogo no tiene nada |
| `top_k` > nº de puntos | Devuelve lo que haya | — |
| Consulta en blanco | `ValueError` | No es *"no encontré nada"*: es que no hay nada que buscar. Codificar `"   "` y devolver diez vecinos sería peor que fallar |
| Marca en blanco | `ValueError` | D14 dejó los huecos como cadena vacía, así que `equals ""` no dejaría de filtrar: devolvería justo los productos **sin marca** |

**Cada caso se ejecuta con tres frases distintas** —palabras clave, la misma necesidad en prosa y una consulta de categoría lejana—, y las dos últimas filas con tres variantes de entrada en blanco: con una sola frase, *"la colección vacía devuelve lista vacía"* sería indistinguible de *"esa consulta no casaba con nada"*.

Los dos primeros y el cuarto necesitan una colección aparte, así que se crea una **vacía y desechable** bajo el prefijo `aurum_humo`, que existe para eso. El tercero se prueba apuntando a un puerto donde no hay nadie: la misma condición que apagar el contenedor, sin apagarlo.

> 🧹 Al terminar se puede borrar desde el panel de Qdrant (<http://localhost:6333/dashboard>): queda vacía y no ocupa nada, pero no es del índice.

In [ ]:
# Colección desechable: prefijo `aurum_humo`, el que existe para esto.
# `recreate=False`, así que no borra nada ni necesita AURUM_ALLOW_RESET.
vacia = QdrantStore(
    collection="aurum_humo_casos_borde",
    url=os.environ.get("AURUM_QDRANT_URL", "http://localhost:6333"),
    api_key=os.environ.get("AURUM_QDRANT_API_KEY"),
    timeout=TIMEOUT_S,
)
vacia.create_collection(dim=DIM, metric="cosine", recreate=False)

# La colección vive en el volumen y sobrevive al notebook: si esto ya se
# ejecutó una vez, dentro están los 3 puntos del último caso borde y el
# primero dejaría de medir una colección vacía —diría "3 resultados" y
# parecería un fallo del sistema en vez de un residuo de la ejecución
# anterior—. Se borran por id, que no necesita AURUM_ALLOW_RESET.
IDS_DESECHABLES = [f"00000000-0000-0000-0000-00000000000{i}" for i in (1, 2, 3)]
for record_id in IDS_DESECHABLES:
    vacia.delete(record_id)

buscador_vacio = BuscadorVectorial(vacia, codificar_consulta, timeout_s=TIMEOUT_S)

# Un motor en un puerto donde no hay nadie: la misma condición que tenerlo
# apagado, sin tener que apagarlo.
# `prefer_grpc=True` es el valor por defecto de QdrantStore, y el cliente
# de gRPC no deriva su puerto de `url` —usa `grpc_port`, que por defecto es
# el 6334 real—. Sin desactivarlo aquí, `.buscar()` ignora el 6399 apagado
# y sale por gRPC contra el Qdrant que sí está levantado: el caso borde no
# mediría nada. `prefer_grpc=False` fuerza a que todo vaya por el `url`
# REST, que es el puerto que de verdad se apagó.
ausente = QdrantStore(
    collection=COLECCION, url="http://localhost:6399",
    prefix=CATALOG_PREFIX, timeout=5, prefer_grpc=False,
)
buscador_ausente = BuscadorVectorial(ausente, codificar_consulta, timeout_s=5)

print(f"colección desechable: {vacia.count()} puntos")

colección desechable: 0 puntos


In [ ]:
# Tres frases por caso: palabras clave, la misma necesidad en prosa, y una
# consulta de categoría lejana. Si el comportamiento dependiera de la frase,
# se vería aquí.
FRASES_BORDE = [
    POR_CASO[caso]
    for caso in ("EVAL-100455-direct", "EVAL-100455-context", "DEV-61533")
]


def poblar_y_pedir_de_mas(consulta):
    """Mete 3 puntos en la desechable y pide 50. Debe devolver 3.

    Va después del caso de la colección vacía y **el orden importa**: si
    esto corriera antes, aquel caso ya no mediría una colección vacía."""
    vacia.upsert([
        Point(
            record_id=record_id,
            vector=codificar_consulta(consulta),
            payload={"product_id": f"TEST-{i}", "title": f"punto de prueba {i}"},
        )
        for i, record_id in enumerate(IDS_DESECHABLES, start=1)
    ], batch_size=3)
    return buscador_vacio.buscar(consulta, top_k=50)


# `partial` y no `lambda`: en un bucle, el lambda capturaría la variable y
# las tres filas acabarían ejecutando la última frase.
casos_borde = []
for frase in FRASES_BORDE:
    casos_borde.append((
        "colección vacía", frase, "lista vacía",
        partial(buscador_vacio.buscar, frase),
    ))
for frase in FRASES_BORDE:
    casos_borde.append((
        "filtro sin resultados", f"{frase} · marca=MarcaQueNoExiste", "lista vacía",
        partial(buscador.buscar, frase, marca="MarcaQueNoExiste"),
    ))
for frase in FRASES_BORDE:
    casos_borde.append((
        "motor no disponible", frase, "MotorNoDisponible, no lista vacía",
        partial(buscador_ausente.buscar, frase),
    ))
# Este puebla la colección desechable, así que va después del caso que la
# necesita vacía.
for frase in FRASES_BORDE:
    casos_borde.append((
        "top_k=50 sobre 3 puntos", frase, "devuelve 3, sin reventar",
        partial(poblar_y_pedir_de_mas, frase),
    ))
# En estos dos la entrada inválida ES el caso, así que la variedad va ahí.
for blanca in ("", "   ", "\n"):
    casos_borde.append((
        "consulta en blanco", repr(blanca), "ValueError: es entrada inválida",
        partial(buscador.buscar, blanca),
    ))
for marca_blanca in ("", "   "):
    casos_borde.append((
        "marca en blanco", f"{FRASES_BORDE[0]} · marca={marca_blanca!r}",
        "ValueError: un filtro vacío dejaría de filtrar",
        partial(buscador.buscar, FRASES_BORDE[0], marca=marca_blanca),
    ))

bordes = auditar_casos_borde(casos_borde)
bordes.style.hide(axis="index").set_properties(
    **{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"}
)

c:\Users\asus\Master\modulos\modulo10_bbdd\practica\AURUM_MARKET\aurum-market-catalog\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:290: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


caso,consulta,esperado,observado,lo_que_devolvio
colección vacía,taladro 24v batería,lista vacía,0 resultados,(lista vacía)
colección vacía,taladro sin cable de 24 voltios que venga con su batería,lista vacía,0 resultados,(lista vacía)
colección vacía,lentejas sin gluten,lista vacía,0 resultados,(lista vacía)
filtro sin resultados,taladro 24v batería · marca=MarcaQueNoExiste,lista vacía,0 resultados,(lista vacía)
filtro sin resultados,taladro sin cable de 24 voltios que venga con su batería · marca=MarcaQueNoExiste,lista vacía,0 resultados,(lista vacía)
filtro sin resultados,lentejas sin gluten · marca=MarcaQueNoExiste,lista vacía,0 resultados,(lista vacía)
motor no disponible,taladro 24v batería,"MotorNoDisponible, no lista vacía",aurum.busqueda.MotorNoDisponible: La base vectorial no respondió. colección 'aurum_catalogo__gemini_embedding_2__A4__768' · qdrant_client.http.exceptions.ResponseHandlingException. Comprueba que el mo,"— (levantó, no devolvió)"
motor no disponible,taladro sin cable de 24 voltios que venga con su batería,"MotorNoDisponible, no lista vacía",aurum.busqueda.MotorNoDisponible: La base vectorial no respondió. colección 'aurum_catalogo__gemini_embedding_2__A4__768' · qdrant_client.http.exceptions.ResponseHandlingException. Comprueba que el mo,"— (levantó, no devolvió)"
motor no disponible,lentejas sin gluten,"MotorNoDisponible, no lista vacía",aurum.busqueda.MotorNoDisponible: La base vectorial no respondió. colección 'aurum_catalogo__gemini_embedding_2__A4__768' · qdrant_client.http.exceptions.ResponseHandlingException. Comprueba que el mo,"— (levantó, no devolvió)"
top_k=50 sobre 3 puntos,taladro 24v batería,"devuelve 3, sin reventar",3 resultados,TEST-2 (1.000) · TEST-3 (1.000) · TEST-1 (1.000)


---

## E · Las comprobaciones de forma

Las cuatro que no dependen del filtro: que `k` se respeta, que no hay `product_id` repetidos dentro de una consulta, que el orden es monótono en la dirección que declara `score_es_similitud`, y que todo lo devuelto existe en el catálogo.

Se ejecutan sobre **las diez consultas del notebook** —las cuatro filtradas y las seis de demostración—, y la tabla enseña el número medido en vez de un `True`:

| Columna | Qué número lleva y cómo se lee |
|---|---|
| `devueltos` | `10 de 10 posibles (pedidos 10)`. Los "posibles" son el oráculo: los productos de esa marca, o los 15.000 puntos si no hay filtro |
| `posiciones` | El rango de `rank`. Tiene que ser `1→n`, sin huecos ni ceros |
| `ids_distintos` | `10 de 10` significa ninguno repetido. Menos a la izquierda es un duplicado dentro de la misma consulta |
| `score_primero_ultimo` | El score del primero y el del último. Enseña de paso **cuánto se aplana** el ranking, que un booleano escondería |
| `orden` | Qué dirección lleva y cuál debería llevar según el motor |
| `fuera_del_catalogo` | Ids recuperados que no existen en el CSV. Cualquier cosa distinta de `0` es un desajuste entre lo indexado y los datos |

La diferencia no es estética: un `k_respetado = True` con `top_k=10` y una marca de 3 productos es correcto y **parece un fallo**, mientras que `3 de 3 posibles (pedidos 10)` se entiende sin mirar el código. Son comprobaciones baratas y aburridas, y son las que impiden que un fallo tonto llegue a `resultados_busqueda.csv` en NB09.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv + las 10 consultas de este notebook
EN_CATALOGO = set(completo["product_id"])
casos_forma = filtradas.to_dict("records") + [
    {"workload_id": caso, "query_text": texto} for caso, texto in DEMO
]

forma = auditar_forma_de_los_resultados(
    buscador, casos_forma,
    ids_del_catalogo=EN_CATALOGO, n_puntos=almacen.count(),
    alcance=ORACULO, top_k=TOP_K,
)
forma.style.hide(axis="index").set_properties(
    **{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"}
)

caso,consulta,marca,devueltos,posiciones,ids_distintos,score_primero_ultimo,orden,fuera_del_catalogo,veredicto
FILTER-001,herramienta inalámbrica para perforar,Einhell,10 de 10 posibles (pedidos 10),1→10,10 de 10,0.5348 → 0.4389,descendente (similitud: ✅),0 de 10,"✅ k, ids, orden y catálogo"
FILTER-002,tableta ligera para estudiar y tomar apuntes,Apple,10 de 10 posibles (pedidos 10),1→10,10 de 10,0.6002 → 0.5757,descendente (similitud: ✅),0 de 10,"✅ k, ids, orden y catálogo"
FILTER-003,zapatillas cómodas para salir a correr,NIKE,10 de 10 posibles (pedidos 10),1→10,10 de 10,0.6004 → 0.5404,descendente (similitud: ✅),0 de 10,"✅ k, ids, orden y catálogo"
FILTER-004,monitor para trabajar con varias ventanas,SAMSUNG,10 de 10 posibles (pedidos 10),1→10,10 de 10,0.6057 → 0.5570,descendente (similitud: ✅),0 de 10,"✅ k, ids, orden y catálogo"
EVAL-100455-direct,taladro 24v batería,— (sin filtro),10 de 10 posibles (pedidos 10),1→10,10 de 10,0.6105 → 0.5691,descendente (similitud: ✅),0 de 10,"✅ k, ids, orden y catálogo"
EVAL-100455-semantic,quiero una herramienta inalámbrica potente p…,— (sin filtro),10 de 10 posibles (pedidos 10),1→10,10 de 10,0.6421 → 0.5998,descendente (similitud: ✅),0 de 10,"✅ k, ids, orden y catálogo"
EVAL-100455-context,taladro sin cable de 24 voltios que venga co…,— (sin filtro),10 de 10 posibles (pedidos 10),1→10,10 de 10,0.6431 → 0.6003,descendente (similitud: ✅),0 de 10,"✅ k, ids, orden y catálogo"
DEV-38249,estantes sin taladro habitacion,— (sin filtro),10 de 10 posibles (pedidos 10),1→10,10 de 10,0.6340 → 0.5675,descendente (similitud: ✅),0 de 10,"✅ k, ids, orden y catálogo"
DEV-43240,funda ipad air 4 sin tapa,— (sin filtro),10 de 10 posibles (pedidos 10),1→10,10 de 10,0.6561 → 0.6165,descendente (similitud: ✅),0 de 10,"✅ k, ids, orden y catálogo"
DEV-61533,lentejas sin gluten,— (sin filtro),10 de 10 posibles (pedidos 10),1→10,10 de 10,0.7239 → 0.6833,descendente (similitud: ✅),0 de 10,"✅ k, ids, orden y catálogo"


---

## F · El artefacto

Las seis tablas juntas en `artifacts/recuperacion.md`, **con los resultados dentro**. Es lo que un corrector necesita para ver que la interfaz cumple §3.3 sin abrir el notebook, y sobre todo para poder discutirla: un resumen de veredictos no se puede contradecir, una lista de `product_id` con su marca y su score sí.

In [ ]:
destino = Path("..") / "artifacts" / "recuperacion.md"
bloques = [
    "# Recuperación · la interfaz común (NB05)",
    "",
    f"Colección: `{COLECCION}` · motor **qdrant** (R03) · `top_k` {TOP_K} · "
    f"timeout {TIMEOUT_S} s · {almacen.count()} puntos",
    "",
    "## Seis consultas y lo que devolvieron",
    "",
    "Una fila por producto recuperado; los 5 primeros de cada consulta.",
    "",
    tabla_de_resultados(demo, top=5).to_markdown(index=False),
    "",
    "### Solapamiento entre las tres formulaciones de la misma necesidad",
    "",
    solapamiento_entre_consultas(demo[:3]).to_markdown(index=False),
    "",
    "## Las cuatro consultas filtradas (§5)",
    "",
    pureza.to_markdown(index=False),
    "",
    "### Los 40 resultados, uno por fila",
    "",
    "Cada consulta pide su marca escrita de otra forma: "
    + ", ".join(f"`{v}`" for v in ESCRITURAS.values()) + ".",
    "",
    detalle_filtradas.to_markdown(index=False),
    "",
    "### La misma marca, escrita de todas las formas en que se escribe",
    "",
    variantes.to_markdown(index=False),
    "",
    "## Filtro nativo contra post-filtro, en las cuatro marcas",
    "",
    "Las marcas van de "
    f"{min(ORACULO.values())} a {max(ORACULO.values())} productos sobre "
    f"{len(completo)}: el post-filtro falla donde la marca es rara.",
    "",
    comparativa.to_markdown(index=False),
    "",
    "## Casos borde (§3.3), cada uno con varias frases",
    "",
    bordes.to_markdown(index=False),
    "",
    "## Comprobaciones de forma",
    "",
    forma.to_markdown(index=False),
    "",
]
destino.write_text("\n".join(bloques), encoding="utf-8")
print(f"Escrito {destino} · {destino.stat().st_size / 1024:.1f} KB")

Escrito ..\artifacts\recuperacion.md · 41.5 KB
